In [ ]:
%matplotlib inline
import numpy as np
import scipy
import statistics
import matplotlib as mpl
from matplotlib import gridspec
import matplotlib.ticker as ticker
from scipy.optimize import curve_fit
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator, FormatStrFormatter)
from scipy import interpolate
import matplotlib.patches as mpatches
import pandas as pd 
from numpy import *
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.signal import find_peaks
import matplotlib.ticker as plticker
# import probfit
from scipy import special
import time
import datetime

mpl.rc('font', family='Arial')

In [ ]:
def readNeutronData(filename):
    tmp = pd.read_csv(filename, sep=',', header = None, skiprows=0)
    return tmp

def _2gaussian(x, amp1,cen1,sigma1, amp2,cen2,sigma2):
    return amp1*(1/(sigma1*(np.sqrt(2*np.pi))))*(np.exp((-1.0/2.0)*(((x_array-cen1)/sigma1)**2))) + \
            amp2*(1/(sigma2*(np.sqrt(2*np.pi))))*(np.exp((-1.0/2.0)*(((x_array-cen2)/sigma2)**2)))

def _1gaussian(x, amp1,cen1,sigma1):
    return amp1*(1/(sigma1*(np.sqrt(2*np.pi))))*(np.exp((-1.0/2.0)*(((x_array-cen1)/sigma1)**2)))


In [ ]:
amp1 = 150
sigma1 = 5
cen1 = 31
amp2 = 250
sigma2 = 5
cen2 = 72

In [ ]:
data = readNeutronData('DataToPlot/VaporWaveIsland.csv')
# x_array = data.iloc[i].index
# y_array_2gauss = data.iloc[i].values

In [ ]:
# popt_2gauss, pcov_2gauss = scipy.optimize.curve_fit(_2gaussian, x_array, y_array_2gauss, p0=[amp1, cen1, sigma1, amp2, cen2, sigma2])
# perr_2gauss = np.sqrt(np.diag(pcov_2gauss))
# pars_1 = popt_2gauss[0:3]
# pars_2 = popt_2gauss[3:6]
# gauss_peak_1 = _1gaussian(x_array, *pars_1)
# gauss_peak_2 = _1gaussian(x_array, *pars_2)

In [ ]:
## gammas on the left!!
## peak 1 = gamma
## peak 2 = neutron

neutron_sigmaValues = []
gamma_sigmaValues = []
neutron_centroids = []
gamma_centroids = []
energy_Slices = []

for i in range(13,14):
    amp1 = 100
    sigma1 = 8
    cen1 = 25
    amp2 = 200
    sigma2 = 10
    cen2 = 72
    
    x_array = data.iloc[i].index
    y_array_2gauss = data.iloc[i].values
    
    popt_2gauss, pcov_2gauss = scipy.optimize.curve_fit(_2gaussian, x_array, y_array_2gauss, p0=[amp1, cen1, sigma1, amp2, cen2, sigma2])
    perr_2gauss = np.sqrt(np.diag(pcov_2gauss))
    pars_1 = popt_2gauss[0:3]
    pars_2 = popt_2gauss[3:6]
    gauss_peak_1 = _1gaussian(x_array, *pars_1)
    gauss_peak_2 = _1gaussian(x_array, *pars_2)

    FOM = abs(popt_2gauss[4] - popt_2gauss[1])/((2.36*(popt_2gauss[2]+popt_2gauss[5])))

    neutron_sigmaValues.append(popt_2gauss[5])
    gamma_sigmaValues.append(popt_2gauss[2])
    neutron_centroids.append(popt_2gauss[4])
    gamma_centroids.append(popt_2gauss[1])
    energy_Slices.append(i*15)
    
    fig, ax = plt.subplots(1, 1, figsize = (7,7))
    fig.tight_layout()
    # ax.set_title('Slice #' + str(i) +' - '+ str(i*15) + ' keVee', fontsize = 16)
    ax.plot(x_array, _2gaussian(x_array, *popt_2gauss), 'k--')#,\
    ax.plot(data.iloc[i].index,data.iloc[i].values)
    ax.plot(x_array, gauss_peak_1, "orange") ## gamma
    ax.set_xlabel('Pulse shape discriminarion (arb. unit)', fontsize = 24)
    ax.set_ylabel('Counts', fontsize = 24)
    ax.fill_between(x_array, gauss_peak_1.min(), gauss_peak_1, facecolor="orange", alpha=0.2)
    ax.plot(x_array, gauss_peak_2, "blue") ## neutron
    ax.fill_between(x_array, gauss_peak_2.min(), gauss_peak_2, facecolor="blue", alpha=0.2)
    ax.tick_params(axis="x", labelsize = 24)
    ax.tick_params(axis="y", labelsize = 24)
    # ax.annotate('FOM = ' + (str(FOM)),(0,10), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
    # ax.annotate('5$\sigma$',(1+popt_2gauss[4]-popt_2gauss[5]*5,250), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14, rotation = 90 )
    # ax.annotate('4$\sigma$',(1+popt_2gauss[4]-popt_2gauss[5]*4,250), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14, rotation = 90)
    # ax.annotate('3$\sigma$',(1+popt_2gauss[4]-popt_2gauss[5]*3,250), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14, rotation = 90 )

    
# plot slices into the same graph
# for i in range(10,11):
#     plt.plot(data.iloc[i].index,data.iloc[i].values, alpha = 0.4)

    plt.axvline(popt_2gauss[4], color = 'black')
    # plt.axvline(popt_2gauss[4] - popt_2gauss[5]*5, ls = ':') # window is centroid of gamma + 5sigma of gammas
    plt.axvline(popt_2gauss[1] + popt_2gauss[2]*5, ls = 'dashdot', color = 'red')
    # plt.axvline(popt_2gauss[4] - popt_2gauss[5]*4, ls = 'dotted', color = 'red') # window is centroid of neutron + 4sigma of neutrons
    # plt.axvline(popt_2gauss[4] - popt_2gauss[5]*3, ls = 'dashed', color = 'red')
    # plt.axvline(popt_2gauss[4] - popt_2gauss[5]*3.5, ls = 'dashdot') # 3.5sigma is a 99.95% confidence / 465.3 ppm (1 om 2149) expected fraction outside of range
    plt.axvline(popt_2gauss[1], color = 'black')


    plt.savefig("Fig3c-PSDSlice.pdf", format="pdf", bbox_inches="tight") 
    
    plt.show()

In [ ]:
df = pd.DataFrame(energy_Slices, columns=['energy(keVee)'])
df['gamma-sigma'] = gamma_sigmaValues
df['gamma-centroid'] = gamma_centroids
df['gFWHM'] = df['gamma-sigma']*2.36
df['n-sigma'] = neutron_sigmaValues
df['neutron-centroid'] = neutron_centroids
df['nFWHM'] = df['n-sigma']*2.36
df['FOM'] = abs(df['gamma-centroid'] - df['neutron-centroid'])/ (df['nFWHM']+df['gFWHM'])
# df['energySlice'] = 
df.head(20)

In [ ]:
plt.plot(df['energy(keVee)'],df['FOM'])
plt.axhline(1.27)
plt.axvline(195)
plt.xlim(0,600)
plt.ylim(0,2)
plt.show()

In [ ]:
print("-------------Peak 1-------------")
print("amplitude = %0.2f (+/-) %0.2f" % (pars_1[0], perr_2gauss[0]))
print("center = %0.2f (+/-) %0.2f" % (pars_1[1], perr_2gauss[1]))
print("sigma = %0.2f (+/-) %0.2f" % (pars_1[2], perr_2gauss[2]))
print("area = %0.2f" % np.trapz(gauss_peak_1))
print("--------------------------------")
print("-------------Peak 2-------------")
print("amplitude = %0.2f (+/-) %0.2f" % (pars_2[0], perr_2gauss[3]))
print("center = %0.2f (+/-) %0.2f" % (pars_2[1], perr_2gauss[4]))
print("sigma = %0.2f (+/-) %0.2f" % (pars_2[2], perr_2gauss[5]))
print("area = %0.2f" % np.trapz(gauss_peak_2))
print("--------------------------------")

In [ ]:
plt.plot(data.iloc[7].index,data.iloc[7].values, alpha = 0.4)
plt.show()

In [ ]:
72.80-35.9

# Gamma sigma line

In [ ]:
neutron_sigmaValues = []
gamma_sigmaValues = []
neutron_centroids = []
gamma_centroids = []
energy_Slices = []

for i in range(4,40):
    amp1 = 100
    sigma1 = 8
    cen1 = 25
    amp2 = 200
    sigma2 = 10
    cen2 = 72
    
    x_array = data.iloc[i].index
    y_array_2gauss = data.iloc[i].values
    
    popt_2gauss, pcov_2gauss = scipy.optimize.curve_fit(_2gaussian, x_array, y_array_2gauss, p0=[amp1, cen1, sigma1, amp2, cen2, sigma2])
    perr_2gauss = np.sqrt(np.diag(pcov_2gauss))
    pars_1 = popt_2gauss[0:3]
    pars_2 = popt_2gauss[3:6]
    gauss_peak_1 = _1gaussian(x_array, *pars_1)
    gauss_peak_2 = _1gaussian(x_array, *pars_2)

    FOM = abs(popt_2gauss[4] - popt_2gauss[1])/((2.36*(popt_2gauss[2]+popt_2gauss[5])))

    neutron_sigmaValues.append(popt_2gauss[5])
    gamma_sigmaValues.append(popt_2gauss[2])
    neutron_centroids.append(popt_2gauss[4])
    gamma_centroids.append(popt_2gauss[1])
    energy_Slices.append(i*15)
    
    fig, ax = plt.subplots(1, 1, figsize = (8,4))
    fig.tight_layout()
    ax.set_title('Slice #' + str(i) +' - '+ str(i*15) + ' keVee')
    ax.plot(x_array, _2gaussian(x_array, *popt_2gauss), 'k--')#,\
    ax.plot(data.iloc[i].index,data.iloc[i].values)
    ax.plot(x_array, gauss_peak_1, "orange") ## gamma
    ax.set_xlabel('Energy (keVee)')
    ax.set_ylabel('Counts')
    ax.fill_between(x_array, gauss_peak_1.min(), gauss_peak_1, facecolor="orange", alpha=0.3)
    ax.plot(x_array, gauss_peak_2, "blue") ## neutron
    ax.fill_between(x_array, gauss_peak_2.min(), gauss_peak_2, facecolor="blue", alpha=0.3)
    ax.annotate('FOM = ' + (str(FOM)),(0,10), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
    ax.annotate('5$\sigma$',(1+popt_2gauss[1]+popt_2gauss[2]*5,250), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14, rotation = 90 )
    ax.annotate('4$\sigma$',(1+popt_2gauss[1]+popt_2gauss[2]*4,250), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14, rotation = 90)
    ax.annotate('3$\sigma$',(1+popt_2gauss[1]+popt_2gauss[2]*3,250), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14, rotation = 90 )

    
# plot slices into the same graph
# for i in range(10,11):
#     plt.plot(data.iloc[i].index,data.iloc[i].values, alpha = 0.4)

    plt.axvline(popt_2gauss[4], color = 'black')
    # plt.axvline(popt_2gauss[4] - popt_2gauss[5]*5, ls = ':') # window is centroid of gamma + 5sigma of gammas
    plt.axvline(popt_2gauss[1] + popt_2gauss[2]*5, ls = 'dashdot', color = 'red')
    plt.axvline(popt_2gauss[1] + popt_2gauss[2]*4, ls = 'dotted', color = 'red') # window is centroid of neutron + 4sigma of neutrons
    plt.axvline(popt_2gauss[1] + popt_2gauss[2]*3, ls = 'dashed', color = 'red')
    # plt.axvline(popt_2gauss[4] - popt_2gauss[5]*3.5, ls = 'dashdot') # 3.5sigma is a 99.95% confidence / 465.3 ppm (1 om 2149) expected fraction outside of range
    plt.axvline(popt_2gauss[1], color = 'black')

    plt.show()